# 04 — Model Training & Scorecard Construction

This notebook trains a **Logistic Regression** model on WoE-transformed features,  
converts predicted probabilities into credit scores (300–900) using **PDO scaling**,  
and assigns each applicant to a risk tier.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import config
from src.scorecard import (
    train_logistic_model, compute_raw_score,
    scale_scores, assign_risk_tier
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

## 4.1 Load WoE Features

In [ ]:
woe_df = pd.read_csv(config.WOE_FEATURES_CSV)
print(f"WoE feature matrix: {woe_df.shape}")
print(f"Columns: {list(woe_df.columns)}")
woe_df.head()

## 4.2 Train / Test Split & Model Training

In [ ]:
target = config.TARGET_COLUMN
feature_cols = [c for c in woe_df.columns if c != target]

X = woe_df[feature_cols]
y = woe_df[target]

model, X_train, X_test, y_train, y_test = train_logistic_model(X, y)

print(f"\nTrain size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test  default rate: {y_test.mean():.2%}")

## 4.3 Model Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c" if c < 0 else "#2ecc71" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors, edgecolor="white")
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_title("Logistic Regression Coefficients (WoE Features)", fontsize=14, fontweight="bold")
ax.set_xlabel("Coefficient")
plt.tight_layout()
plt.show()

print(f"\nIntercept: {model.intercept_[0]:.4f}")
print(coef_df.to_string(index=False))

## 4.4 Classification Report (Test Set)

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Fully Paid", "Charged Off"]))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Fully Paid", "Charged Off"],
            yticklabels=["Fully Paid", "Charged Off"])
ax.set_title("Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 4.5 PDO Score Scaling

Convert predicted probabilities → raw scores → scaled 300–900 scores.

**Formula:** `Score = Base_Score − PDO × log₂(odds / Base_Odds)`  
Then min-max scale to [300, 900].

In [ ]:
proba = model.predict_proba(X)[:, 1]
raw_scores = compute_raw_score(proba)
credit_scores = scale_scores(raw_scores)

print(f"Raw  scores — mean: {raw_scores.mean():.1f}, std: {raw_scores.std():.1f}")
print(f"Scaled scores — min: {credit_scores.min():.0f}, max: {credit_scores.max():.0f}, mean: {credit_scores.mean():.0f}")

## 4.6 Credit Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(credit_scores, bins=60, color="#3498db", edgecolor="white", alpha=0.85)

# Mark tier boundaries
for tier, (low, high) in config.RISK_TIERS.items():
    ax.axvline(x=low, color="#e74c3c", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_title("Credit Score Distribution (300–900)", fontsize=14, fontweight="bold")
ax.set_xlabel("Credit Score")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 4.7 Risk Tier Assignment

In [ ]:
tiers = pd.Series(credit_scores).apply(lambda s: assign_risk_tier(int(round(s))))
tier_counts = tiers.value_counts()

tier_colors = {
    "Super Prime": "#27ae60",
    "Prime": "#2ecc71",
    "Near Prime": "#f1c40f",
    "Subprime": "#e67e22",
    "Deep Subprime": "#e74c3c",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
order = ["Super Prime", "Prime", "Near Prime", "Subprime", "Deep Subprime"]
ordered_counts = tier_counts.reindex(order).fillna(0)
colors = [tier_colors.get(t, "#95a5a6") for t in ordered_counts.index]
axes[0].bar(ordered_counts.index, ordered_counts.values, color=colors, edgecolor="white")
axes[0].set_title("Applications by Risk Tier", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Count")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

# Pie chart
axes[1].pie(ordered_counts.values, labels=ordered_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, textprops={"fontsize": 10})
axes[1].set_title("Portfolio Concentration", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nRisk Tier Distribution:")
print(ordered_counts.to_string())

## 4.8 Summary

- Logistic Regression trained on WoE features with 70/30 stratified split.
- Predicted probabilities converted to credit scores via PDO scaling.
- Scores scaled to 300–900 range and assigned to 5 risk tiers.
- Model coefficients align with credit intuition (higher WoE for FICO → lower default risk).

Proceed to **05_evaluation.ipynb** →